In [ ]:
# Find the project root regardless of where Jupyter's working directory is set
# (works whether VS Code launches the kernel from notebooks/ or the project root).
import os
from pathlib import Path

def find_project_root(marker="run_pipeline.py"):
    p = Path.cwd()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root (looking for {marker})")

ROOT = find_project_root()
os.chdir(ROOT)
print("Working directory set to:", ROOT)

# 02 — Validate Local Output vs. Production

Checks that our **local DuckDB run** behaves like the **real production output**
(Manish's `august_2026_recommendations.csv`). This is a one-time sanity check —
"does our local pipeline reproduce what the warehouse actually does?" Run it
once after any change to the runner itself (translation logic, source loading).

For "did MY methodology change break anything?", use `03_compare_to_baseline.ipynb`
instead — that compares against your own saved baseline, not August's file.

**What it checks:** row/clinic counts, the hardcoded caps (14 samples, education
gating), scenario/action/classification mix, opportunity distribution, and the
top-seller targeting metric.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None); pd.set_option('display.width', 160)

LOCAL = "output/recommendations_local.csv"
# Point this at wherever you keep Manish's real production export:
PRODUCTION = "data/august_2026_recommendations.csv"   # adjust path if needed

local = pd.read_csv(LOCAL, low_memory=False)
local.columns = [c.strip() for c in local.columns]
prod = pd.read_csv(PRODUCTION, low_memory=False)
prod.columns = [c.strip() for c in prod.columns]

print(f"LOCAL:      {local.shape[0]:,} rows x {local.shape[1]} cols  | month: {local['SCHEDULED MONTH'].iloc[0]}")
print(f"PRODUCTION: {prod.shape[0]:,} rows x {prod.shape[1]} cols  | month: {prod['SCHEDULED MONTH'].iloc[0]}")
print(f"\nSame columns: {sorted(local.columns) == sorted(prod.columns)}")
print("(Different SCHEDULED MONTH is expected — local uses whatever data you loaded.)")

## 1. Shape — clinics, rows-per-clinic, TMs

In [ ]:
def shape_check(df, label):
    print(f"--- {label} ---")
    print("clinics:", df['CLINIC ID'].nunique())
    rpc = df.groupby('CLINIC ID').size()
    print(f"rows/clinic: min {rpc.min()} max {rpc.max()} (expect 26)")
    print("TMs:", df['CLINIC SALES REP OR TM'].nunique())
    print()

shape_check(local, "LOCAL")
shape_check(prod, "PRODUCTION")

## 2. The hardcoded caps — sample cap (14) & education gating

In [ ]:
def cap_check(df, label):
    print(f"--- {label} ---")
    sampled = df[df['HAS SAMPLE']==True].groupby('CLINIC SALES REP OR TM')['CLINIC ID'].nunique()
    print("Max samples for any TM (cap=14):", sampled.max() if len(sampled) else 0,
          "| any TM over 14?", (sampled > 14).any() if len(sampled) else False)
    edu = df[df['DISEASE_CATEGORY_EDUCATION_FLAG']==1]
    print(f"Education recs: {len(edu)} ({len(edu)/len(df)*100:.2f}% of rows)")
    print()

cap_check(local, "LOCAL")
cap_check(prod, "PRODUCTION")

## 3. Scenario, action & classification mix — side by side

In [ ]:
for col in ['SCENARIO - CLINIC DISEASE CATEGORY', 'ACTION RECOMMENDATION']:
    comp = pd.DataFrame({
        'local_%': (local[col].value_counts(normalize=True)*100).round(1),
        'prod_%':  (prod[col].value_counts(normalize=True)*100).round(1),
    }).fillna(0)
    print(f"--- {col} ---")
    print(comp)
    print()

In [ ]:
comp = pd.DataFrame({
    'local_%': (local['DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION'].value_counts(normalize=True)*100).round(1),
    'prod_%':  (prod['DISEASE_CATEGORY_OPPORTUNITY_CLASSIFICATION'].value_counts(normalize=True)*100).round(1),
}).fillna(0)
print("--- DC OPPORTUNITY CLASSIFICATION ---")
print(comp.sort_values('local_%', ascending=False))

## 4. Opportunity distribution (\$ and the volume gap)

In [ ]:
def opp_check(df, label):
    opp = pd.to_numeric(df['MONTHLY OPPORTUNITY'], errors='coerce').fillna(0)
    vol = pd.to_numeric(df['MONTHLY OPPORTUNITY VOLUME'], errors='coerce').fillna(0)
    print(f"--- {label} ---")
    print(f"$ opportunity: mean {opp.mean():.2f}  median {opp.median():.2f}  max {opp.max():.2f}  | zero rows: {(opp==0).mean()*100:.1f}%")
    print(f"volume opportunity: non-zero rows {(vol!=0).mean()*100:.2f}%")
    print()

opp_check(local, "LOCAL")
opp_check(prod, "PRODUCTION")

## 5. The real quality metric — top recommendation vs. top-selling category

⚠️ Don't use `TOP DISEASE CATEGORY IN CLINIC FLAG` — it's circular (just marks
opportunity-rank #1). This compares the #1 *recommendation* to the clinic's
highest-*share* category instead.

In [ ]:
def top_seller_rate(df):
    rank1 = df[df['OPPORTUNITY RANKING WITHIN CLINIC']==1][['CLINIC ID','DISEASE CATEGORY','SPECIES']]
    top_share = (df.sort_values('DISEASE_CATEGORY_SHARE_PCT', ascending=False)
                   .groupby('CLINIC ID').first()[['DISEASE CATEGORY','SPECIES']]
                   .rename(columns={'DISEASE CATEGORY':'top_share_dc','SPECIES':'top_share_sp'}))
    chk = rank1.merge(top_share, on='CLINIC ID')
    chk['same'] = (chk['DISEASE CATEGORY']==chk['top_share_dc']) & (chk['SPECIES']==chk['top_share_sp'])
    return chk['same'].mean()*100

print(f"LOCAL:      #1 rec == top-selling category  {top_seller_rate(local):.1f}% of clinics")
print(f"PRODUCTION: #1 rec == top-selling category  {top_seller_rate(prod):.1f}% of clinics")
print("(Lower = better targeting of real gaps, not the obvious category.)")

## Verdict

If the local numbers land in the same ballpark as production across all 5 sections
(caps identical, distributions similar shape, top-seller rate close), the local
runner is trustworthy and you can build on it with confidence.

Small differences are EXPECTED because local uses whatever data extract you loaded
(likely a different month / same underlying logic). Large differences (wrong cap,
wildly different opportunity magnitude, a classification category missing entirely)
mean something in the runner's Snowflake→DuckDB translation needs another look.